In [1]:
from yumi_jacobi import YuMi_EGM as YuMi
from jacobi import Planner, Frame, CartesianWaypoint, LinearMotion

yumi = YuMi()

/home/mallika/anaconda3/envs/lipnew/lib/python3.8/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


[jacobi.driver] Timeout while waiting for EGM session.
[jacobi.driver] Timeout while waiting for EGM session.


In [2]:
import time

print('Left current position', yumi.driver_left.current_joint_position)
print('Right current position', yumi.driver_right.current_joint_position)

# Move both arms one after the other
home_left = [-0.967, -0.734, 0.748, -0.118, 1.365, 1.477, -1.6]
home_right = [0.740, -0.319, -0.876, -0.310, -0.866, 1.276, -1.8]

yumi.driver_left.move_to(home_left)
yumi.driver_right.move_to(home_right)

# Calculate tcp position in Task space
print(yumi.left.calculate_tcp(home_left))
print(yumi.right.calculate_tcp(home_right))

# Move both arms synchronized
trajectory = yumi.planner.plan(
    start={
        yumi.left: yumi.driver_left.current_joint_position,
        yumi.right: yumi.driver_right.current_joint_position,
    },
    goal={
        yumi.left: CartesianWaypoint(Frame(x=0.60, y=0.24, z=0.25, b=-3.1415, c=1.85)),
        yumi.right: CartesianWaypoint(Frame(x=0.50, y=-0.14, z=0.25, b=-3.1415, c=1.12)),
    },
)

result_left = yumi.driver_left.run_async(trajectory)
result_right = yumi.driver_right.run_async(trajectory)
await result_left
await result_right

# Linear motion of left arm
motion = LinearMotion(
    yumi.left,
    yumi.driver_left.current_joint_position,
    Frame(x=0.60, y=0.14, z=0.25, b=-3.1415, c=1.0),
)
motion.robot.set_speed(0.05)

tl = yumi.planner.plan(motion)
result_left = yumi.driver_left.run_async(tl)

# And stop after a timeout because we can control the robot on-the-fly
time.sleep(3.0)
yumi.driver_left.stop()

# Back to home
yumi.set_speed(0.15)
result_left = yumi.driver_left.move_to_async(home_left)
result_right = yumi.driver_right.move_to_async(home_right)
await result_left
await result_right


JacobiError: 
[jacobi.exception.driver]
	The driver is not connected to the Robot via EGM.


In [2]:

yumi.calibrate_grippers()

In [2]:
yumi.gripper_right.open_gripper()

In [3]:
yumi.gripper_left.open_gripper()

In [7]:
yumi.close_grippers()

In [9]:
yumi.open_grippers()

In [10]:
yumi.gripper_left.get_gripper_state()

Signal(cmd_GripperState_L, 5)